# **Predictive Modeling**

The goal of this file is to create a predictive model from Los Angeles Angels (LAA) data to estimate the expected price change in Kalshi sports trading markets.

## Data Description

### Features

- **d_price_w $\alpha$ _b $\beta$**: The double binned mean price at the time of a home run.
    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **laa_homerun_dummy**: Binary indicator equal to 1 if LAA hit a home run at that timestamp, 0 otherwise.

- **runners_on_base**: The number of runners on base when the home run was hit.

- **inning**: The inning in which the home run was hit.

- **laa_score**: LAA's score when the home run was hit.

- **opp_score**: The opponent's score when the home run was hit.

- **score_delta**: LAA's score less the opponent's score when the home run was hit.

- **opponent_XYZ**: Opponent dummy variable; 1 if that team is playing LAA, 0 otherwise.

    - _XYZ_ is the abbreviated team name of the team LAA is playing.

- **d_rolling_std_ $\gamma$ trades_w $\alpha$ _b $\beta$**: The rolling standard deviation of percent price change over a window defined by trade count. $$\sigma_{\gamma} = \sqrt{\frac{1}{\gamma - 1} \sum_{i=1}^{\gamma} \left( p_{i} - \overline{p} \right)^2}$$

    - $\gamma$ is the trade count which determines the window size: $\gamma \in \{2, 3, 4, \ldots, 49, 50\}$
    
    - Corresponds with the double binned mean percent price change and price
        - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
        - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

- **d_rolling_std_ $\lambda$ mins_w $\alpha$ _b $\beta$**: The rolling standard deviation of percent price change over a window defined by time (minutes). $$\sigma_{\lambda} = \sqrt{\frac{1}{n_\lambda - 1} \sum_{i=1}^{n_\lambda} \left( p_{i} - \overline{p} \right)^2}$$

    - $\lambda$ is the number of minutes which determines the window size: $\lambda \in \{2, 3, 4, \ldots, 49, 50\}$
    - $n_\lambda$ is the number of $p_{i}$ observations falling within the trailing $\lambda$-minute window.

    - Corresponds with the double binned mean percent price change and price
        - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
        - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

### Targets

- **d_pct_px_chg_w $\alpha$ _b $\beta$**: The double binned mean percent price change measuring market reaction to a home run.
    - $\alpha$ is the window size: $\alpha \in \{1, 2, 3, 4, 5, 6\}$
    - $\beta$ is the bin size: $\beta \in \{1, 2, 3, 4, 5, 6\}$

In [233]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
from scipy.stats import norm
from datetime import timedelta
import plotly.graph_objects as go
from utils.data_handlers import DataProcessing

In [234]:
LAA_RED = '#BA0021'
LAA_NAVY = '#003263'
LAA_SILVER = '#C4CED4'

## NGBoost Model for Percent Price Change Prediction

In [235]:
# Load in all NGBoost results across different double binned targets

ngb_dir_path = "../models/ngb/"

ngb_results = {}
for w in range(1, 7):
    for b in range(1, 7):
        file_name = f"model_d_pct_px_chg_w{w}_b{b}"
        path = Path(ngb_dir_path) / f"{file_name}.pkl"
        try:
            with open(path, 'rb') as f:
                ngb_results[file_name] = pickle.load(f)
        except FileNotFoundError:
            print(f"Missing: {path}")

In [236]:
# Get summary statistics for plotting

ngb_model_stats = []
for k, v in ngb_results.items():
    ngb_model_stats.append({
        'y_col'       : k,
        'window'      : k.split('_')[-2][1],
        'bin'         : k.split('_')[-1][1],
        'test_r2'     : ngb_results[k]['best_test_r2'],
        'overfitting' : ngb_results[k]['best_overfitting']
    })

ngb_model_stats_df = pd.DataFrame(ngb_model_stats)

ngb_model_stats_df.sort_values('test_r2', ascending=False).head()

,y_col,window,bin,test_r2,overfitting
21,model_d_pct_px_chg_w4_b4,4,4,0.846923,0.060429
11,model_d_pct_px_chg_w2_b6,2,6,0.840784,0.105712
16,model_d_pct_px_chg_w3_b5,3,5,0.835606,0.090878
26,model_d_pct_px_chg_w5_b3,5,3,0.814776,0.067613
31,model_d_pct_px_chg_w6_b2,6,2,0.808545,0.080839


In [237]:
# Plotting NGBoost models across all window and bin combinations

# Drop extreme outliers (four dropped)
ngb_model_stats_df = ngb_model_stats_df[ngb_model_stats_df['test_r2'] > 0.33]

laa_color_scale = [
    [0.0, LAA_NAVY],
    [0.5, LAA_SILVER],
    [1.0, LAA_RED]
]

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=ngb_model_stats_df['window'],
    y=ngb_model_stats_df['bin'],
    z=ngb_model_stats_df['test_r2'],
    mode='markers',
    marker=dict(
        size=8,
        color=ngb_model_stats_df['overfitting'],
        colorscale=laa_color_scale,
        opacity=0.9,
        colorbar=dict(
            title=dict(
                text='Test R² - Train R²',
                font=dict(color=LAA_SILVER)
            ),
            tickfont=dict(color=LAA_SILVER)
        ),
        showscale=True
    ),
    text=[
        f"Window: {w}<br>Bin: {b}<br>Test R²: {r:.4f}<br>Overfitting: {m:.4f}" 
        for w, b, r, m in zip(
            ngb_model_stats_df['window'], 
            ngb_model_stats_df['bin'], 
            ngb_model_stats_df['test_r2'], 
            ngb_model_stats_df['overfitting']
            )
    ],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    title=dict(
        text='<b>Optimally Parameterized NGBoost Models (LAA)</b>',
        font={'size': 25, 'color': LAA_SILVER},
        x=0.5,
        xanchor='center',
        y=0.95,        
        yanchor='top' 
    ),
    scene=dict(
        xaxis_title='Window Size (α)',
        yaxis_title='Bin Size (β)',
        zaxis_title='Test R²',
        xaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        yaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        zaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        bgcolor='#111111',
        aspectmode='cube'
    ),
    width=1000,
    height=700,
    margin=dict(l=0, r=0, b=0, t=60),
    showlegend=False
)

fig.show()

### Parameter Analysis for a Strong Performing NGBoost Model

In [238]:
# Get strong performing NGB model, uses target with window = 4 and bin = 4

strong_ngb = ngb_results["model_d_pct_px_chg_w4_b4"]

In [239]:
# Strong NGB statistics

print('Train MSE:', strong_ngb['best_train_mse'])
print('Test MSE:', strong_ngb['best_test_mse'])
print('Train R²:', strong_ngb['best_train_r2'])
print('Test R²:', strong_ngb['best_test_r2'])

Train MSE: 0.027899565891217046
Test MSE: 0.04450289751957216
Train R²: 0.9073516977494598
Test R²: 0.8469229690380204


In [240]:
# Grid Search CV results for target with window = 4 and bin = 4

gscv_cols = [
    'rank_test_score',
    'param_learning_rate', 
    'param_minibatch_frac', 
    'param_n_estimators', 
    'mean_test_score', 
    'std_test_score',
    'mean_train_score', 
    'std_train_score'
]

strong_ngb['cv_results'][gscv_cols].sort_values('rank_test_score')

,rank_test_score,param_learning_rate,param_minibatch_frac,param_n_estimators,mean_test_score,std_test_score,mean_train_score,std_train_score
53,1,0.05,1.0,150,3.305663,4.087698,-1.094587,0.207697
50,2,0.05,0.8,150,3.210398,4.717631,-1.154955,0.165814
47,3,0.05,0.5,150,2.260941,3.269076,-0.996683,0.107048
49,4,0.05,0.8,100,1.483971,2.508955,-0.943492,0.147040
52,5,0.05,1.0,100,1.482915,2.025412,-0.913948,0.180683
46,6,0.05,0.5,100,1.262292,2.368154,-0.789053,0.107741
26,7,0.05,1.0,150,0.928825,1.339315,-0.687161,0.135903
23,8,0.05,0.8,150,0.879742,1.229130,-0.718478,0.099933
20,9,0.05,0.5,150,0.866928,1.452620,-0.658027,0.082320
6,10,0.01,1.0,50,0.551793,0.168428,0.452103,0.055826


In [241]:
# Strong NGB parameters

strong_ngb['best_params']

{'Base': DecisionTreeRegressor(criterion='friedman_mse', max_depth=3, min_samples_leaf=8,
                       min_samples_split=15),
 'Dist': ngboost.distns.normal.Normal,
 'learning_rate': 0.05,
 'minibatch_frac': 1.0,
 'n_estimators': 150}

### Feature Importance for a Strong Performing NGBoost Model

In [242]:
# Get feature importance

strong_ngb_model = strong_ngb['best_estimator']

feature_importance_loc = strong_ngb_model.feature_importances_[0]    # Mean
feature_importance_scale = strong_ngb_model.feature_importances_[1]  # Standard deviation

In [243]:
# Create feature importance data frames

X_test_df = pd.DataFrame(strong_ngb['X_test']).reset_index(drop=True)

features = X_test_df.columns

loc_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance_loc * 100
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

scale_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance_scale * 100
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

In [244]:
# Plot feature importance for location

fig_loc = px.bar(
    loc_df.head(10), 
    x='importance', 
    y='feature', 
    orientation='h',
    title='<b>Strong NGB Top 10 Feature Importances: Location (μ)</b><br><span style="font-size:15px; color:' + LAA_SILVER + '">Target Percent Price Change Window = 4 & Bin = 4</span>',
    color_discrete_sequence=[LAA_RED]
)

fig_loc.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    yaxis={'categoryorder':'total ascending'},
    title_font={'size': 22, 'color': LAA_SILVER},
    xaxis_title='Relative Feature Importance (%)',
    yaxis_title=None,
    margin=dict(l=20, r=20, t=60, b=40)
)

fig_loc.show()

In [245]:
# Plot feature importance for scale

fig_scale = px.bar(
    scale_df.head(10), 
    x='importance', 
    y='feature', 
    orientation='h',
    title='<b>Strong NGB Top 10 Feature Importances: Scale (σ)</b><br><span style="font-size:15px; color:' + LAA_SILVER + '">Target Percent Price Change Window = 4 & Bin = 4</span>',
    color_discrete_sequence=[LAA_NAVY]
)

fig_scale.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    yaxis={'categoryorder':'total ascending'},
    title_font={'size': 22, 'color': LAA_SILVER},
    xaxis_title='Relative Feature Importance (%)',
    yaxis_title=None,
    margin=dict(l=20, r=20, t=60, b=40)
)

fig_scale.show()

## XGBoost Model for Percent Price Change Prediction

In [246]:
# Load in all XGBoost results across different double binned targets

xgb_dir_path = "../models/xgb/"

xgb_results = {}
for w in range(1, 7):
    for b in range(1, 7):
        file_name = f"model_d_pct_px_chg_w{w}_b{b}"
        path = Path(xgb_dir_path) / f"{file_name}.pkl"
        try:
            with open(path, 'rb') as f:
                xgb_results[file_name] = pickle.load(f)
        except FileNotFoundError:
            print(f"Missing: {path}")

In [247]:
# Get summary statistics for plotting

xgb_model_stats = []
for k, v in xgb_results.items():
    xgb_model_stats.append({
        'y_col'       : k,
        'window'      : k.split('_')[-2][1],
        'bin'         : k.split('_')[-1][1],
        'test_r2'     : xgb_results[k]['best_test_r2'],
        'overfitting' : xgb_results[k]['best_overfitting']
    })

xgb_model_stats_df = pd.DataFrame(xgb_model_stats)

xgb_model_stats_df.sort_values('test_r2', ascending=False).head()

,y_col,window,bin,test_r2,overfitting
21,model_d_pct_px_chg_w4_b4,4,4,0.744936,-0.063031
31,model_d_pct_px_chg_w6_b2,6,2,0.730211,-0.153048
26,model_d_pct_px_chg_w5_b3,5,3,0.722015,-0.124304
13,model_d_pct_px_chg_w3_b2,3,2,0.708009,0.004663
16,model_d_pct_px_chg_w3_b5,3,5,0.700764,-0.009930


In [248]:
# Plotting XGBoost models across all window and bin combinations

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=xgb_model_stats_df['window'],
    y=xgb_model_stats_df['bin'],
    z=xgb_model_stats_df['test_r2'],
    mode='markers',
    marker=dict(
        size=8,
        color=xgb_model_stats_df['overfitting'],
        colorscale=laa_color_scale,
        opacity=0.9,
        colorbar=dict(
            title=dict(
                text='Test R² - Train R²',
                font=dict(color=LAA_SILVER)
            ),
            tickfont=dict(color=LAA_SILVER)
        ),
        showscale=True
    ),
    text=[
        f"Window: {w}<br>Bin: {b}<br>Test R²: {r:.4f}<br>Overfitting: {m:.4f}" 
        for w, b, r, m in zip(
            xgb_model_stats_df['window'], 
            xgb_model_stats_df['bin'], 
            xgb_model_stats_df['test_r2'], 
            xgb_model_stats_df['overfitting']
            )
    ],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    title=dict(
        text='<b>Optimally Parameterized XGBoost Models (LAA)</b>',
        font={'size': 25, 'color': LAA_SILVER},
        x=0.5,
        xanchor='center',
        y=0.95,        
        yanchor='top' 
    ),
    scene=dict(
        xaxis_title='Window Size (α)',
        yaxis_title='Bin Size (β)',
        zaxis_title='Test R²',
        xaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        yaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        zaxis=dict(gridcolor='rgb(60, 60, 60)', showbackground=False),
        bgcolor='#111111',
        aspectmode='cube'
    ),
    width=1000,
    height=700,
    margin=dict(l=0, r=0, b=0, t=60),
    showlegend=False
)

fig.show()

### Parameter Analysis for a Strong Performing XGBoost Model

In [249]:
# Get strong performing XGB model, uses target with window = 4 and bin = 4 
# Note, top-performing XGBoost model matches top-performing NGBoost model!

strong_xgb = xgb_results["model_d_pct_px_chg_w4_b4"]

In [250]:
# Strong XGB statistics

print('Train MSE:', strong_xgb['best_train_mse'])
print('Test MSE:', strong_xgb['best_test_mse'])
print('Train R²:', strong_xgb['best_train_r2'])
print('Test R²:', strong_xgb['best_test_r2'])

Train MSE: 0.09578921808645466
Test MSE: 0.07415284639036483
Train R²: 0.6819051427459457
Test R²: 0.7449357638381944


In [251]:
# Random Search CV results for target with window = 4 and bin = 4

rscv_cols = [
    'rank_test_score',
    'param_reg_lambda',   
    'param_reg_alpha', 
    'param_n_estimators', ''
    'param_min_child_weight', 
    'param_max_depth',    
    'param_learning_rate',
    'param_gamma',        
    'param_colsample_bytree', 
    'mean_train_score',   
    'std_train_score',
    'mean_test_score', 
    'std_test_score',
]

strong_xgb['cv_results'][rscv_cols].sort_values('rank_test_score')

,rank_test_score,param_reg_lambda,param_reg_alpha,param_n_estimators,param_min_child_weight,param_max_depth,param_learning_rate,param_gamma,param_colsample_bytree,mean_train_score,std_train_score,mean_test_score,std_test_score
23,1,10,1,200,5,3,0.30,0.5,0.7,-0.190902,0.008754,-0.221153,0.040313
37,2,50,1,100,10,3,0.10,0.5,0.5,-0.208384,0.006790,-0.222769,0.048182
28,2,50,1,100,5,2,0.10,0.5,0.5,-0.208384,0.006790,-0.222769,0.048182
47,4,10,1,50,10,2,0.30,1.0,0.7,-0.210304,0.007949,-0.227122,0.044140
34,5,100,1,100,5,2,0.30,0.5,0.5,-0.220613,0.005937,-0.228356,0.049902
41,6,10,5,200,10,3,0.10,0.5,0.7,-0.218201,0.005250,-0.229098,0.050405
4,7,10,1,200,15,3,0.01,1.0,0.7,-0.220544,0.006950,-0.233656,0.048946
42,8,10,1,200,10,2,0.30,2.0,0.7,-0.226504,0.003548,-0.238240,0.054414
30,9,50,5,100,10,3,0.10,0.5,0.7,-0.232103,0.004112,-0.239900,0.053357
8,10,50,1,100,5,3,0.05,1.0,0.7,-0.232864,0.004883,-0.241366,0.053294


In [252]:
# Strong XGB parameters

strong_xgb['best_params']

{'reg_lambda': 10,
 'reg_alpha': 1,
 'n_estimators': 200,
 'min_child_weight': 5,
 'max_depth': 3,
 'learning_rate': 0.3,
 'gamma': 0.5,
 'colsample_bytree': 0.7}

### Feature Importance for a Strong Performing XGBoost Model

In [253]:
# Get feature importance

strong_xgb_model = strong_xgb['best_estimator']

feature_importance = strong_xgb_model.feature_importances_

In [255]:
# Create feature importance data frames

X_test_df = pd.DataFrame(strong_xgb['X_test']).reset_index(drop=True)

features = X_test_df.columns

feature_importance_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance * 100
    }
).sort_values('importance', ascending=False).reset_index(drop=True)

In [256]:
# Plot feature importance

fig = px.bar(
    feature_importance_df.head(10), 
    x='importance', 
    y='feature', 
    orientation='h',
    title='<b>Strong XGB Top 10 Feature Importances</b><br><span style="font-size:15px; color:' + LAA_SILVER + '">Target Percent Price Change Window = 4 & Bin = 4</span>',
    color_discrete_sequence=[LAA_RED]
)

fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111',
    yaxis={'categoryorder':'total ascending'},
    title_font={'size': 22, 'color': LAA_SILVER},
    xaxis_title='Relative Feature Importance (%)',
    yaxis_title=None,
    margin=dict(l=20, r=20, t=60, b=40)
)

fig.show()

## Model Performance on Real-World Examples

### Example One: The Angels Hit a Homer

[On June 14, 2025, the Angels (LAA) played the Orioles (BAL).](https://www.espn.com/mlb/game/_/gameId/401695951/angels-orioles)

It's the 1st inning and the game is tied zero to zero. Mike Trout is up to bat with one runner on base. Checking your phone, you see that Kalshi's market for the game is trading at ~46% (LAA has a ~46% chance of winning). You look up, and Trout has just hit a home run! You wonder what the market will be trading at now?

In [257]:
# Get the specific LAA vs BAL game from the test set

laa_vs_bal = X_test_df.loc[78, :] # Index of the specific game

In [ ]:
# Percent price change according to the double binned method

px_at_hr = laa_vs_bal['d_price_w4_b4']

dbm_pct_px_chg = float(strong_ngb['y_test'][78]) # Index of the specific game

print(f"Double Binned Mean Price Before Home Run: {px_at_hr*100:.0f}")
print(f"Double Binned Mean Percent Price Change: {dbm_pct_px_chg*100:.2f}%")
print(f"Double Binned Mean Price After Home Run: {100 * px_at_hr*(1+dbm_pct_px_chg):.0f}")

Double Binned Mean Price Before Home Run: 46
Double Binned Mean Percent Price Change: 34.95%
Double Binned Mean Price After Home Run: 63


### Analysis Using NGBoost

In [259]:
# Predict the percent price change distribution for the home run

laa_vs_bal_dist = strong_ngb_model.pred_dist(laa_vs_bal.values.reshape(1, -1))

ngb_mean_pct_px_chg = laa_vs_bal_dist.params['loc'][0]
ngb_std_pct_px_chg = laa_vs_bal_dist.params['scale'][0]

print(f"Strong NGB Expected Percent Price Change: {ngb_mean_pct_px_chg*100:.2f}%")
print(f"Strong NGB Standard Deviation Percent Price Change: {ngb_std_pct_px_chg*100:.2f}%")
print(f"Strong NGB Expected Price on Kalshi: {px_at_hr*(1+ngb_mean_pct_px_chg)*100 :.0f}")

Strong NGB Expected Percent Price Change: 29.81%
Strong NGB Standard Deviation Percent Price Change: 9.38%
Strong NGB Expected Price on Kalshi: 60


In [260]:
# Confidence intervals for percent price change

z99 = norm.ppf(0.995)
z95 = norm.ppf(0.975)

ci99 = np.array((
    ngb_mean_pct_px_chg - ngb_std_pct_px_chg * z99,
    ngb_mean_pct_px_chg + ngb_std_pct_px_chg * z99
))
ci95 = np.array((
    ngb_mean_pct_px_chg - ngb_std_pct_px_chg * z95,
    ngb_mean_pct_px_chg + ngb_std_pct_px_chg * z95
))

price99 = px_at_hr * (1 + ci99) * 100
price95 = px_at_hr * (1 + ci95) * 100

print("Strong NGB Expected Percent Price Change:")
print(f"99% CI: [{ci99[0]*100:.2f}%, {ci99[1]*100:.2f}%]")
print(f"95% CI: [{ci95[0]*100:.2f}%, {ci95[1]*100:.2f}%]")
print()
print("Strong NGB Expected Price on Kalshi:")
print(f"99% CI: [{price99[0]:.0f}, {price99[1]:.0f}]")
print(f"95% CI: [{price95[0]:.0f}, {price95[1]:.0f}]")

Strong NGB Expected Percent Price Change:
99% CI: [5.66%, 53.96%]
95% CI: [11.44%, 48.19%]

Strong NGB Expected Price on Kalshi:
99% CI: [49, 72]
95% CI: [52, 69]


### Analysis Using XGBoost

In [261]:
# Predict the percent price change for the home run

xgb_pct_px_chg = strong_xgb_model.predict(laa_vs_bal.values.reshape(1, -1))

print(f"Strong XGB Expected Percent Price Change: {xgb_pct_px_chg[0]*100:.2f}%")
print(f"Strong XGB Expected Price on Kalshi: {px_at_hr*(1+xgb_pct_px_chg[0])*100 :.0f}")

Strong XGB Expected Percent Price Change: 31.00%
Strong XGB Expected Price on Kalshi: 61


### Real-World Outcome

In [ ]:
# Instantiate DataProcessing object for LAA

laa_DP = DataProcessing(
    team = 'LAA',
    trade_path = "../data/raw/laa_kalshi_trade_data.parquet",
    score_path = "../data/raw/laa_score_data.parquet",
    homerun_path = "../data/raw/laa_homerun_data.parquet"
)

laa_trade_df = laa_DP.get_trade_df()

In [263]:
# Load LAA vs BAL June 14, 2025 game trades

laa_vs_bal_ticker = "KXMLBGAME-25JUN14LAABAL-LAA"
laa_vs_bal_trades_df = laa_trade_df[laa_trade_df['ticker'] == laa_vs_bal_ticker].reset_index(drop=True).copy()

laa_vs_bal_trades_df.head()

,ticker,time_pst,price,trade_count
0,KXMLBGAME-25JUN14LAABAL-LAA,2025-06-14 05:06:56.583700-07:00,0.42,1
1,KXMLBGAME-25JUN14LAABAL-LAA,2025-06-14 05:11:16.250810-07:00,0.42,22
2,KXMLBGAME-25JUN14LAABAL-LAA,2025-06-14 05:35:37.145020-07:00,0.42,23
3,KXMLBGAME-25JUN14LAABAL-LAA,2025-06-14 06:10:09.287662-07:00,0.42,350
4,KXMLBGAME-25JUN14LAABAL-LAA,2025-06-14 06:19:21.760431-07:00,0.42,1


In [ ]:
# Plot LAA vs BAL June 14, 2025 game

# Approximate home run time according to double binned mean method
laa_vs_bal_trades_df['time_pst'] = pd.to_datetime(laa_vs_bal_trades_df['time_pst'])
homerun_time = pd.to_datetime('2025-06-14 13:09:52.072145-07:00')

# Prepare data for plotting
start_time = homerun_time - timedelta(minutes=15)
end_time = homerun_time + timedelta(minutes=15)

game_window = laa_vs_bal_trades_df[
    (laa_vs_bal_trades_df['time_pst'] >= start_time) & 
    (laa_vs_bal_trades_df['time_pst'] <= end_time)
].copy()

game_window['price'] = game_window['price'] * 100

hr_price = game_window.loc[game_window['time_pst'] == homerun_time, 'price'].iloc[0]

# Plotting
fig = px.line(game_window, x='time_pst', y='price', color_discrete_sequence=[LAA_RED])

fig.update_layout(
    template='plotly_dark',
    title={
        'text': "<b>Kalshi Market Reaction to Mike Trout's Home Run</b><br><span style='font-size:16px; color:" + LAA_SILVER + "'>LAA vs BAL | June 14, 2025</span>",
        'font': {'size': 20, 'color': LAA_SILVER},
        'x': 0.5, 'xanchor': 'center'
    },
    xaxis_title='Time (PST)',
    yaxis_title='Price (Probability of LAA Winning)',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111'
)

fig.add_annotation(
    x=homerun_time,
    y=hr_price,
    text="<b>HR is Hit</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor='white',
    font=dict(color='white', size=12),
    bgcolor='rgba(186, 0, 33, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2
)

fig.add_annotation(
    text=f"<b>Expected Price on Kalshi After Home Run</b><br>" + 
         f"Strong NGB: {px_at_hr * (1 + ngb_mean_pct_px_chg) * 100 :.0f} <br>" +
         f"Strong XGB: {px_at_hr * (1 + xgb_pct_px_chg[0]) * 100 :.0f}",
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    xanchor='left', yanchor='top',
    showarrow=False,
    font=dict(size=12, color='white'),
    bgcolor='rgba(17, 17, 17, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2,
    borderpad=10
)

fig.show()

### Example Two: An Angels Opponent Hits a Homer

[On August 15, 2025, the Angels (LAA) played the A's (ATH).](https://www.espn.com/mlb/game/_/gameId/401764579/angels-athletics) 

It's the 3rd inning and the game is tied 1-1. Colby Thomas is up to bat with two runners on base. Checking your phone, you see that Kalshi's market for the game is trading at ~35% (LAA has a ~35% chance of winning). You look up, and Thomas has just hit a home run! You wonder what the market will be trading at now?

In [265]:
# Get the specific LAA vs ATH game from the test set

laa_vs_ath = X_test_df.loc[2, :] # Index of the specific game

In [ ]:
# Percent price change according to the double binned method

px_at_hr = laa_vs_ath['d_price_w4_b4']

dbm_pct_px_chg = float(strong_ngb['y_test'][2]) # Index of the specific game

print(f"Double Binned Mean Price Before Home Run: {px_at_hr*100:.0f}")
print(f"Double Binned Mean Percent Price Change: {dbm_pct_px_chg*100:.2f}%")
print(f"Double Binned Mean Price After Home Run: {100*px_at_hr*(1+dbm_pct_px_chg):.0f}")

Double Binned Mean Price Before Home Run: 35
Double Binned Mean Percent Price Change: -54.61%
Double Binned Mean Price After Home Run: 16


### Analysis Using NGBoost

In [267]:
# Predict the percent price change distribution for the home run

laa_vs_ath_dist = strong_ngb_model.pred_dist(laa_vs_ath.values.reshape(1, -1))

ngb_mean_pct_px_chg = laa_vs_ath_dist.params['loc'][0]
ngb_std_pct_px_chg = laa_vs_ath_dist.params['scale'][0]

print(f"Strong NGB Expected Percent Price Change: {ngb_mean_pct_px_chg*100:.2f}%")
print(f"Strong NGB Standard Deviation Percent Price Change: {ngb_std_pct_px_chg*100:.2f}%")
print(f"Strong NGB Expected Price on Kalshi: {px_at_hr*(1+ngb_mean_pct_px_chg)*100:.0f}")

Strong NGB Expected Percent Price Change: -48.48%
Strong NGB Standard Deviation Percent Price Change: 8.49%
Strong NGB Expected Price on Kalshi: 18


In [268]:
# Confidence intervals for percent price change

ci99 = np.array((
    ngb_mean_pct_px_chg - ngb_std_pct_px_chg * z99,
    ngb_mean_pct_px_chg + ngb_std_pct_px_chg * z99
))
ci95 = np.array((
    ngb_mean_pct_px_chg - ngb_std_pct_px_chg * z95,
    ngb_mean_pct_px_chg + ngb_std_pct_px_chg * z95
))

price99 = px_at_hr * (1 + ci99) * 100
price95 = px_at_hr * (1 + ci95) * 100

print("Strong NGB Expected Percent Price Change:")
print(f"99% CI: [{ci99[0]*100:.2f}%, {ci99[1]*100:.2f}%]")
print(f"95% CI: [{ci95[0]*100:.2f}%, {ci95[1]*100:.2f}%]")
print()
print("Strong NGB Expected Price on Kalshi:")
print(f"99% CI: [{price99[0]:.0f}, {price99[1]:.0f}]")
print(f"95% CI: [{price95[0]:.0f}, {price95[1]:.0f}]")

Strong NGB Expected Percent Price Change:
99% CI: [-70.35%, -26.60%]
95% CI: [-65.12%, -31.83%]

Strong NGB Expected Price on Kalshi:
99% CI: [10, 26]
95% CI: [12, 24]


### Analysis Using XGBoost

In [269]:
# Predict the percent price change for the home run

xgb_pct_px_chg = strong_xgb_model.predict(laa_vs_ath.values.reshape(1, -1))

print(f"Strong XGB Expected Percent Price Change: {xgb_pct_px_chg[0]*100:.2f}%")
print(f"Strong XGB Expected Price on Kalshi: {px_at_hr*(1+xgb_pct_px_chg[0])*100 :.0f}")

Strong XGB Expected Percent Price Change: -35.67%
Strong XGB Expected Price on Kalshi: 23


### Real-World Outcome

In [270]:
# Load LAA vs ATH August 15, 2025 game trades

laa_vs_ath_ticker = "KXMLBGAME-25AUG15LAAATH-LAA"
laa_vs_ath_trades_df = laa_trade_df[laa_trade_df['ticker'] == laa_vs_ath_ticker].reset_index(drop=True).copy()

laa_vs_ath_trades_df.head()

,ticker,time_pst,price,trade_count
0,KXMLBGAME-25AUG15LAAATH-LAA,2025-08-15 05:26:33.419343-07:00,0.53,115
1,KXMLBGAME-25AUG15LAAATH-LAA,2025-08-15 06:29:52.524401-07:00,0.52,50
2,KXMLBGAME-25AUG15LAAATH-LAA,2025-08-15 07:35:10.701263-07:00,0.52,37
3,KXMLBGAME-25AUG15LAAATH-LAA,2025-08-15 07:51:42.211188-07:00,0.52,186
4,KXMLBGAME-25AUG15LAAATH-LAA,2025-08-15 08:15:08.684102-07:00,0.52,18


In [ ]:
# Plot LAA vs ATH August 15, 2025 game

# Approximate home run time according to double binned mean method
laa_vs_ath_trades_df['time_pst'] = pd.to_datetime(laa_vs_ath_trades_df['time_pst'])
homerun_time = pd.to_datetime('2025-08-15 20:04:01.239315-07:00')

# Prepare data for plotting
start_time = homerun_time - timedelta(minutes=15)
end_time = homerun_time + timedelta(minutes=15)

game_window = laa_vs_ath_trades_df[
    (laa_vs_ath_trades_df['time_pst'] >= start_time) & 
    (laa_vs_ath_trades_df['time_pst'] <= end_time)
].copy()

game_window['price'] = game_window['price'] * 100

hr_price = game_window.loc[game_window['time_pst'] == homerun_time, 'price'].iloc[0]

# Plotting
fig = px.line(game_window, x='time_pst', y='price', color_discrete_sequence=[LAA_RED])

fig.update_layout(
    template='plotly_dark',
    title={
        'text': "<b>Kalshi Market Reaction to Colby Thomas's Home Run</b><br><span style='font-size:16px; color:" + LAA_SILVER + "'>LAA vs ATH | August 15, 2025</span>",
        'font': {'size': 20, 'color': LAA_SILVER},
        'x': 0.5, 'xanchor': 'center'
    },
    xaxis_title='Time (PST)',
    yaxis_title='Price (Probability of LAA Winning)',
    paper_bgcolor='#111111',
    plot_bgcolor='#111111'
)

fig.add_annotation(
    x=homerun_time,
    y=hr_price,
    text="<b>HR is Hit</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor='white',
    font=dict(color='white', size=12),
    bgcolor='rgba(186, 0, 33, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2
)

fig.add_annotation(
    text=f"<b>Expected Price on Kalshi After Home Run</b><br>" + 
         f"Strong NGB: {px_at_hr * (1 + ngb_mean_pct_px_chg) * 100 :.0f} <br>" +
         f"Strong XGB: {px_at_hr * (1 + xgb_pct_px_chg[0]) * 100 :.0f}",
    xref="paper", yref="paper",
    x=0.02, y=0.40,
    xanchor='left', yanchor='top',
    showarrow=False,
    font=dict(size=12, color='white'),
    bgcolor='rgba(17, 17, 17, 0.8)',
    bordercolor=LAA_SILVER,
    borderwidth=2,
    borderpad=10
)

fig.show()

## Summary of Predictive Modeling

### Target

- Recall, the Double Binned Mean method was used to determine the target being predicted in this analysis. The reader may wonder, why? This method solves two problems.

    - Problem One: Data Quality

        - The window parameter, denoted $\alpha$, determines the number of trades to go out prior or after a home run was claimed to have been hit by `mlbstatsapi`. From the [data analysis](https://github.com/henrycosentino/mlb_analysis/blob/main/notebooks/data_analysis.ipynb) section it was determined that `mlbstatsapi` provides lagged data, with home runs and scores matched to timestamps after they occurred. The window parameter corrects this issue.

    - Problem Two: Kalshi Market Liquidity

        - The bin parameter, denoted $\beta$, determines the number of trades to include in the first window (price window before the home run was hit) and the second window (price window after the home run was hit). Simply taking one trade before and after a home run may result in a poor representation of the market price. Instead, an average could capture a fair market price.

- This approach is further justified by the results in the [data analysis](https://github.com/henrycosentino/mlb_analysis/blob/main/notebooks/data_analysis.ipynb) and [hypothesis testing](https://github.com/henrycosentino/mlb_analysis/blob/main/notebooks/hypothesis_testing.ipynb) notebooks.

- The models trained using **d_pct_px_chg_w4_b4** as the target was chosen for further analysis due to its strong performance in both NGBoost and XGBoost models.

### NGBoost Models

- NGBoost models displayed strong performance with the top five best performing models by test $R^2$ ranging from 0.81 to 0.85, and overfitting ranging from 0.06 to 0.10 for those same models (measured by train $R^2$ minus test $R^2$).

- The parameters selected for the model using **d_pct_px_chg_w4_b4** via Grid Search Cross Validation were:

    - Distribution: Normal

    - Base: Decision Tree Regressor

        - Criterion: Friedman's MSE

        - Max Depth: 3

        - Minimum Samples per Leaf: 8

        - Minimum Samples per Split: 15

    - Learning Rate: 0.05

    - Minibatch Fraction: 1.0

    - Number of Trees (estimators): 150

### XGBoost Models

- XGBoost models displayed strong performance with the top five best performing models by test $R^2$ ranging from 0.70 to 0.74, and overfitting ranging from -0.12 to 0.00 for those same models (measured by train $R^2$ minus test $R^2$). Noticeably less overfitting than NGBoost models, which can be attributed to the regularization imposed on XGBoost models.

- The parameters selected for the model using **d_pct_px_chg_w4_b4** via Random Search Cross Validation were:

    - L1 Regularization (alpha): 1

    - L2 Regularization (lambda): 10

    - Minimum Loss Reduction for Further Partitioning (gamma): 0.5

    - Column Subsample by Tree: 0.7

    - Minimum Child Weight: 5

    - Maximum Tree Depth: 3

    - Learning Rate: 0.3

    - Number of Trees (estimators): 200